In [ ]:
#파일 열어서 시간처리 변환 삼각함수로
import pandas as pd
import numpy as np

input_file = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\2025_seoul_road_data_first.csv"
output_file = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\2025_seoul_road_data_first_clean.csv"

df = pd.read_csv(input_file, encoding='cp949')
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print("첫 번째 완료!")

input_file = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\2025_seoul_road_data_last.csv"
output_file = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\2025_seoul_road_data_last_clean.csv"

df = pd.read_csv(input_file, encoding='cp949')
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print("전체 완료!")

input_file = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\2025_seoul_road_data_first_clean.csv"
output_file = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\2025_seoul_road_data_first_encoded.csv"

# 🔥 dtype=str로 읽어서 속도 + 안정성 확보
df = pd.read_csv(input_file, dtype=str)

# =========================
# 1. 날짜 처리 (A열 = 0번)
# =========================
date_col = df.iloc[:, 0].astype(str)

# 🔥 ".0" 제거
date_col = date_col.str.replace('.0', '', regex=False)

# datetime 변환
date_dt = pd.to_datetime(date_col, format='%Y%m%d', errors='coerce')

# day of year (1~365)
day_of_year = date_dt.dt.dayofyear

# sin / cos 변환
df['day_sin'] = np.sin(2 * np.pi * day_of_year / 365)
df['day_cos'] = np.cos(2 * np.pi * day_of_year / 365)

# =========================
# 2. 시간 처리 (F열 = 5번)
# =========================
time_col = df.iloc[:, 5].astype(int)

# HHMM → hour 변환
hour = time_col // 100
minute = time_col % 100

hour_float = hour + minute / 60.0

# sin / cos 변환
df['hour_sin'] = np.sin(2 * np.pi * hour_float / 24)
df['hour_cos'] = np.cos(2 * np.pi * hour_float / 24)

df['day_sin'] = df['day_sin'].round(6)
df['day_cos'] = df['day_cos'].round(6)
df['hour_sin'] = df['hour_sin'].round(6)
df['hour_cos'] = df['hour_cos'].round(6)

# =========================
# 3. 저장
# =========================
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print("1 변환 완료!")

input_file = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\2025_seoul_road_data_last_clean.csv"
output_file = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\2025_seoul_road_data_last_encoded.csv"

# 🔥 dtype=str로 읽어서 속도 + 안정성 확보
df = pd.read_csv(input_file, dtype=str)

# =========================
# 1. 날짜 처리 (A열 = 0번)
# =========================
date_col = df.iloc[:, 0].astype(str)

# 🔥 ".0" 제거
date_col = date_col.str.replace('.0', '', regex=False)

# datetime 변환
date_dt = pd.to_datetime(date_col, format='%Y%m%d', errors='coerce')

# day of year (1~365)
day_of_year = date_dt.dt.dayofyear

# sin / cos 변환
df['day_sin'] = np.sin(2 * np.pi * day_of_year / 365)
df['day_cos'] = np.cos(2 * np.pi * day_of_year / 365)

# =========================
# 2. 시간 처리 (F열 = 5번)
# =========================

time_col = pd.to_numeric(df.iloc[:, 5], errors='coerce')

# 🔥 이상값 제거
valid_mask = time_col.notna()
df = df[valid_mask]
time_col = time_col[valid_mask]

# HHMM → hour 변환
hour = time_col // 100
minute = time_col % 100

hour_float = hour + minute / 60.0

df['hour_sin'] = np.sin(2 * np.pi * hour_float / 24)
df['hour_cos'] = np.cos(2 * np.pi * hour_float / 24)

df['day_sin'] = df['day_sin'].round(6)
df['day_cos'] = df['day_cos'].round(6)
df['hour_sin'] = df['hour_sin'].round(6)
df['hour_cos'] = df['hour_cos'].round(6)

# =========================
# 3. 저장
# =========================
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print("2 변환 완료!")

첫 번째 완료!
전체 완료!
1 변환 완료!
2 변환 완료!


In [ ]:

#각 구간별로 그룹화
import pandas as pd
import os

input_file1 = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\2025_seoul_road_data_first_encoded.csv"
input_file2 = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\2025_seoul_road_data_last_encoded.csv"

output_dir = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\output_groups"
os.makedirs(output_dir, exist_ok=True)

chunk_size = 100000
written_files = set()

def process_file(input_file):
    csv_iter = pd.read_csv(
        input_file,
        chunksize=chunk_size,
        dtype=str,
        encoding='utf-8-sig',
        engine='python',
        sep=',',
        quotechar='"',
        on_bad_lines='skip'
    )

    for chunk in csv_iter:
        
        # 열 깨짐 방지
        if len(chunk.columns) < 6:
            continue

        col_B = chunk.columns[1]  # B열
        col_D = chunk.columns[3]  # C열

        # 🔥 B + C 둘 다 기준으로 그룹핑
        for (val_B, val_D), group in chunk.groupby([col_B, col_D], sort=False):
            
            # 파일 이름 안전하게
            safe_B = str(val_B).replace('/', '_').replace('\\', '_').replace(' ', '_')
            safe_D = str(val_D).replace('/', '_').replace('\\', '_').replace(' ', '_')

            file_path = os.path.join(output_dir, f"{safe_D}.csv")

            if file_path not in written_files:
                group.to_csv(file_path, index=False, mode='w', encoding='utf-8-sig')
                written_files.add(file_path)
            else:
                group.to_csv(file_path, index=False, mode='a', header=False, encoding='utf-8-sig')


print("파일 1 처리 시작")
process_file(input_file1)

print("파일 2 처리 시작")
process_file(input_file2)

print("완료!")

파일 1 처리 시작
파일 2 처리 시작
완료!


In [ ]:
# 이전 구간의 속도 데이터 열 만들기
import os
import pandas as pd
import re

folder_path = r"D:\\엔지니어링 공모전\\data_set\\road_data_file\\output_groups"

files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

def parse_filename(filename):
    match = re.match(r"([A-Za-z]{3,4})(\d+)\.csv", filename)
    if match:
        prefix = match.group(1)
        number = int(match.group(2))
        return prefix, number
    return None, None

# 📌 그룹화
groups = {}
for file in files:
    prefix, number = parse_filename(file)
    if prefix is not None:
        groups.setdefault(prefix, []).append((file, number))

# 📌 처리
for prefix, file_list in groups.items():

    # 🔥 조건부 정렬
    if prefix == "LLL":
        sorted_files = sorted(file_list, key=lambda x: x[1], reverse=True)  # 내림차순
    else:
        sorted_files = sorted(file_list, key=lambda x: x[1])  # 오름차순

    prev_df = None

    for filename, number in sorted_files:
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path, encoding='utf-8-sig')

        if prev_df is not None:
            try:
                prev_H = prev_df.iloc[:, 7]
                df['Past SPD'] = prev_H.reindex(df.index)

                # 🔥 NaN 행 제거
                df = df.dropna(subset=['Past SPD'])
            except Exception as e:
                print(f"에러 발생: {filename} → {e}")

        df.to_csv(file_path, index=False, encoding='utf-8-sig')

        prev_df = df

print("완료!")

KeyboardInterrupt: 

In [ ]:
#사고관련 데이터 정규화 과정
import pandas as pd

input_file = "D:\\엔지니어링 공모전\\data_set\\accident_data.csv"
output_file = "D:\\엔지니어링 공모전\\data_set\\accident_data_normalized.csv"

print("시작!")

df = pd.read_csv(input_file, encoding='utf-8-sig')

df_norm = df.copy()

# 🔥 컬럼 정의
day_cols = ['일','월','화','수','목','금','토']
month_cols = [f"{i}월" for i in range(1, 13)]
hour_cols = [f"{i:02d}" for i in range(24)]

# 🔥 해당 컬럼들만 float 변환 (핵심 해결)
df_norm[day_cols + month_cols + hour_cols] = \
    df_norm[day_cols + month_cols + hour_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)

# =========================
# 정규화
# =========================
for idx, row in df_norm.iterrows():
    
    # 요일
    max_day = row[day_cols].max()
    if max_day != 0:
        df_norm.loc[idx, day_cols] = row[day_cols] / max_day
    else:
        df_norm.loc[idx, day_cols] = 0

    # 월
    max_month = row[month_cols].max()
    if max_month != 0:
        df_norm.loc[idx, month_cols] = row[month_cols] / max_month
    else:
        df_norm.loc[idx, month_cols] = 0

    # 시간
    max_hour = row[hour_cols].max()
    if max_hour != 0:
        df_norm.loc[idx, hour_cols] = row[hour_cols] / max_hour
    else:
        df_norm.loc[idx, hour_cols] = 0

print("정규화 완료!")

df_norm.to_csv(output_file, index=False, encoding='utf-8-sig')

print("파일 저장 완료!")

시작!
정규화 완료!
파일 저장 완료!


In [14]:
# 전처리한 사고데이터를 각 폴더의 구간별 파일에 추가
import pandas as pd
import os

# ===== 경로 설정 =====
accident_file = "D:\\엔지니어링 공모전\\data_set\\accident_data_normalized.csv"
target_folder = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\output_groups"
output_folder = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\output_groups_accident"

os.makedirs(output_folder, exist_ok=True)

# ===== 사고 데이터 로드 =====
acc_df = pd.read_csv(accident_file, dtype=str)

# 도로 코드 기준으로 빠르게 찾기 위해 index 설정
acc_df.set_index(acc_df.columns[0], inplace=True)

# ===== 각 CSV 파일 처리 =====
for file in os.listdir(target_folder):
    if not file.endswith(".csv"):
        continue

    file_path = os.path.join(target_folder, file)

    # 파일명 앞 3~4글자 추출
    road_code = file[:4] if file[:4] in acc_df.index else file[:3]

    if road_code not in acc_df.index:
        print(f"❌ 매칭 안됨: {file}")
        continue

    print(f"✅ 처리중: {file} ({road_code})")

    df = pd.read_csv(file_path, dtype=str)

    # ===== 날짜 → 요일, 월 추출 =====
    df['date'] = pd.to_datetime(df.iloc[:, 0], format='%Y%m%d', errors='coerce')

    df['weekday'] = df['date'].dt.weekday  # 0=월, 6=일
    df['month'] = df['date'].dt.month      # 1~12

    # ===== 시간 처리 (F열 기준) =====
    # F열이 5번째 index (0부터 시작하면 5)
    df['hour'] = pd.to_numeric(df.iloc[:, 5], errors='coerce') // 60

    # ===== 사고 데이터 가져오기 =====
    acc_row = acc_df.loc[road_code]

    # 요일: B~H (7개)
    weekday_values = acc_row.iloc[0:7].astype(float).values

    # 월: I~T (12개)
    month_values = acc_row.iloc[7:19].astype(float).values

    # 시간: U~AR (24개)
    hour_values = acc_row.iloc[19:43].astype(float).values

    # ===== 값 매핑 =====
    def get_weekday_val(x):
        if pd.isna(x) or x < 0 or x > 6:
            return 0
        return weekday_values[int(x)]

    def get_month_val(x):
        if pd.isna(x) or x < 1 or x > 12:
            return 0
        return month_values[int(x) - 1]

    def get_hour_val(x):
        if pd.isna(x) or x < 0 or x > 23:
            return 0
        return hour_values[int(x)]

    df['acc_weekday'] = df['weekday'].apply(get_weekday_val)
    df['acc_month'] = df['month'].apply(get_month_val)
    df['acc_hour'] = df['hour'].apply(get_hour_val)
    
    # 🔥 불필요한 컬럼 제거
    #df.drop(columns=['date', 'weekday', 'month', 'hour'], inplace=True)

    # ===== 저장 =====
    output_path = os.path.join(output_folder, file)
    df.to_csv(output_path, index=False, encoding='utf-8-sig')

print("🎯 모든 파일 처리 완료!")

✅ 처리중: LBL2000010.csv (LBL)
✅ 처리중: LBL2000020.csv (LBL)
✅ 처리중: LBL2000030.csv (LBL)
✅ 처리중: LBL2000040.csv (LBL)
✅ 처리중: LBL2000050.csv (LBL)
✅ 처리중: LBR2000010.csv (LBR)
✅ 처리중: LBR2000020.csv (LBR)
✅ 처리중: LBR2000030.csv (LBR)
✅ 처리중: LBR2000040.csv (LBR)
✅ 처리중: LBR2000050.csv (LBR)
✅ 처리중: LDI2000010.csv (LDI)
✅ 처리중: LDI2000015.csv (LDI)
✅ 처리중: LDI2000020.csv (LDI)
✅ 처리중: LDI2000025.csv (LDI)
✅ 처리중: LDI2000030.csv (LDI)
✅ 처리중: LDI2000040.csv (LDI)
✅ 처리중: LDI2000050.csv (LDI)
✅ 처리중: LDI2000060.csv (LDI)
✅ 처리중: LDI2000070.csv (LDI)
✅ 처리중: LDI2000083.csv (LDI)
✅ 처리중: LDI2000085.csv (LDI)
✅ 처리중: LDI2000090.csv (LDI)
✅ 처리중: LDI2000110.csv (LDI)
✅ 처리중: LDI2000120.csv (LDI)
✅ 처리중: LDI2000130.csv (LDI)
✅ 처리중: LDO2000010.csv (LDO)
✅ 처리중: LDO2000015.csv (LDO)
✅ 처리중: LDO2000020.csv (LDO)
✅ 처리중: LDO2000030.csv (LDO)
✅ 처리중: LDO2000050.csv (LDO)
✅ 처리중: LDO2000060.csv (LDO)
✅ 처리중: LDO2000070.csv (LDO)
✅ 처리중: LDO2000080.csv (LDO)
✅ 처리중: LDO2000090.csv (LDO)
✅ 처리중: LDO2000100.csv (LDO)
✅ 처리중: LDO2000110.cs

In [15]:
# 날씨 데이터 더하기
import pandas as pd
import glob
import os

# 기준이 되는 CSV (붙일 파일)
base_file = "D:\\엔지니어링 공모전\\data_set\\2025_seoul_rain_clean.csv"
base_df = pd.read_csv(base_file)

# 여러 CSV들이 있는 폴더
input_folder = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\output_groups_accident"
output_folder = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\output_accident_weather"
os.makedirs(output_folder, exist_ok=True)

# 모든 CSV 파일 가져오기
csv_files = glob.glob(os.path.join(input_folder, "*.csv"))

for file in csv_files:
    df = pd.read_csv(file)

    # 🔥 열 방향으로 합치기
    merged = pd.concat([df, base_df], axis=1)

    # 🔥 A열 기준으로 NaN 있는 행만 삭제
    merged_clean = merged.dropna(subset=[merged.columns[0]])

    # 저장
    filename = os.path.basename(file)
    output_path = os.path.join(output_folder, filename)

    merged_clean.to_csv(output_path, index=False, encoding='utf-8-sig')



print("완료!")

완료!


In [ ]:
#공휴일 데이터 더하기
import pandas as pd
import os
import glob

input_folder = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\output_accident_weather"
output_folder = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\output_accident_weater_holiday"
os.makedirs(output_folder, exist_ok=True)

chunk_size = 100000

# 🔥 공휴일 리스트
holiday_list = [
    "20250101","20250128","20250129","20250130",
    "20250301","20250303",
    "20250505","20250506",
    "20250606",
    "20250815",
    "20251003","20251006","20251007","20251008","20251009",
    "20251225"
]

csv_files = glob.glob(os.path.join(input_folder, "*.csv"))

for file in csv_files:
    print(f"처리 중: {file}")
    
    output_file = os.path.join(output_folder, os.path.basename(file))
    first_chunk = True

    for chunk in pd.read_csv(file, chunksize=chunk_size, dtype=str, encoding='utf-8-sig'):
        
        # 🔥 A열 (날짜)
        date_col = chunk.columns[0]

        # 🔥 날짜 문자열 정리
        chunk[date_col] = (
            chunk[date_col]
            .astype(str)
            .str.replace(r"\.0$", "", regex=True)
            .str.strip()
        )

        # 🔥 공휴일 여부만 추가
        chunk["is_holiday"] = chunk[date_col].isin(holiday_list).astype(int)

        # 🔥 저장
        chunk.to_csv(
            output_file,
            mode='w' if first_chunk else 'a',
            index=False,
            header=first_chunk,
            encoding='utf-8-sig'
        )

        first_chunk = False

    print(f"완료: {output_file}")

print("전체 완료!")

처리 중: D:\엔지니어링 공모전\data_set\road_data_file\output_accident_weather\LBL2000010.csv
완료: D:\엔지니어링 공모전\data_set\road_data_file\output_accident_weater_holiday\LBL2000010.csv
처리 중: D:\엔지니어링 공모전\data_set\road_data_file\output_accident_weather\LBL2000020.csv
완료: D:\엔지니어링 공모전\data_set\road_data_file\output_accident_weater_holiday\LBL2000020.csv
처리 중: D:\엔지니어링 공모전\data_set\road_data_file\output_accident_weather\LBL2000030.csv
완료: D:\엔지니어링 공모전\data_set\road_data_file\output_accident_weater_holiday\LBL2000030.csv
처리 중: D:\엔지니어링 공모전\data_set\road_data_file\output_accident_weather\LBL2000040.csv
완료: D:\엔지니어링 공모전\data_set\road_data_file\output_accident_weater_holiday\LBL2000040.csv
처리 중: D:\엔지니어링 공모전\data_set\road_data_file\output_accident_weather\LBL2000050.csv
완료: D:\엔지니어링 공모전\data_set\road_data_file\output_accident_weater_holiday\LBL2000050.csv
처리 중: D:\엔지니어링 공모전\data_set\road_data_file\output_accident_weather\LBR2000010.csv
완료: D:\엔지니어링 공모전\data_set\road_data_file\output_accident_weater_holiday\L

In [17]:
#이전 속도 관련 데이터가 없는 파일을 삭제시켜줌
import os
import pandas as pd

folder_path = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\output_accident_weater_holiday"

for file in os.listdir(folder_path):
    if not file.endswith(".csv"):
        continue

    file_path = os.path.join(folder_path, file)

    try:
        # 🔥 헤더만 읽기 (속도 빠름)
        df = pd.read_csv(file_path, nrows=0)

        # 🔥 열 체크
        if "Past SPD" not in df.columns:
            os.remove(file_path)
            print(f"삭제됨: {file}")
        else:
            print(f"유지됨: {file}")

    except Exception as e:
        print(f"에러 발생 ({file}): {e}")

삭제됨: LBL2000010.csv
유지됨: LBL2000020.csv
유지됨: LBL2000030.csv
유지됨: LBL2000040.csv
유지됨: LBL2000050.csv
삭제됨: LBR2000010.csv
유지됨: LBR2000020.csv
유지됨: LBR2000030.csv
유지됨: LBR2000040.csv
유지됨: LBR2000050.csv
삭제됨: LDI2000010.csv
유지됨: LDI2000015.csv
유지됨: LDI2000020.csv
유지됨: LDI2000025.csv
유지됨: LDI2000030.csv
유지됨: LDI2000040.csv
유지됨: LDI2000050.csv
유지됨: LDI2000060.csv
유지됨: LDI2000070.csv
유지됨: LDI2000083.csv
유지됨: LDI2000085.csv
유지됨: LDI2000090.csv
유지됨: LDI2000110.csv
유지됨: LDI2000120.csv
유지됨: LDI2000130.csv
삭제됨: LDO2000010.csv
유지됨: LDO2000015.csv
유지됨: LDO2000020.csv
유지됨: LDO2000030.csv
유지됨: LDO2000050.csv
유지됨: LDO2000060.csv
유지됨: LDO2000070.csv
유지됨: LDO2000080.csv
유지됨: LDO2000090.csv
유지됨: LDO2000100.csv
유지됨: LDO2000110.csv
유지됨: LDO2000120.csv
유지됨: LDO2000130.csv
유지됨: LDO2000140.csv
유지됨: LDO2000145.csv
삭제됨: LFD2000010.csv
유지됨: LFD2000020.csv
유지됨: LFD2000030.csv
유지됨: LFD2000040.csv
삭제됨: LFU2000010.csv
유지됨: LFU2000020.csv
유지됨: LFU2000030.csv
삭제됨: LHD2000160.csv
삭제됨: LHU2000160.csv
삭제됨: LKI2000090.csv


In [ ]:
#각종 자료 전처리 과정인데 여기서 하는것보다 모델 제작하는 과정에서 하는게 좋음 따라서 그냥 
#만들어 두기만 하고 직접 실행하지는 않는다.

import pandas as pd
import numpy as np
import glob
import os
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# =========================
# 🔥 경로 설정
# =========================
input_folder = "D:\\엔지니어링 공모전\\data_set\\road_data_file\\output_accident_weater_holiday"
output_folder = "D:\\엔지니어링 공모전\\data_set\\output_incoding"
os.makedirs(output_folder, exist_ok=True)

csv_files = glob.glob(os.path.join(input_folder, "*.csv"))

# =========================
# 🔥 컬럼 위치 정의
# =========================
# (0부터 시작)
COL_IDX = {
    "road": 1,        # B
    "section": 2,     # C
    "traffic": 4,     # E
    "speed": 5,       # F
    "rain": 11,       # L
    "snow": 12,       # M
    "visibility": 13  # N
}

# =========================
# 🔥 1단계: 전체 데이터로 fit
# =========================
print("=== 전체 데이터 스케일 기준 생성 ===")

all_data = []

for file in csv_files:
    df = pd.read_csv(file, dtype=str)
    all_data.append(df)

full_df = pd.concat(all_data, ignore_index=True)

# 컬럼 이름 매핑
cols = {k: full_df.columns[v] for k, v in COL_IDX.items()}

# 숫자형 변환
for key in ["traffic", "speed", "rain", "snow", "visibility"]:
    full_df[cols[key]] = pd.to_numeric(full_df[cols[key]], errors='coerce').fillna(0)

# 🔥 log 변환
full_df[cols["rain"]] = np.log1p(full_df[cols["rain"]])
full_df[cols["snow"]] = np.log1p(full_df[cols["snow"]])
full_df[cols["traffic"]] = np.log1p(full_df[cols["traffic"]])

# 🔥 스케일러 fit
scaler_minmax_rain = MinMaxScaler()
scaler_minmax_vis = MinMaxScaler()
scaler_std = StandardScaler()

scaler_minmax_rain.fit(full_df[[cols["rain"], cols["snow"]]])
scaler_minmax_vis.fit(full_df[[cols["visibility"]]])
scaler_std.fit(full_df[[cols["traffic"], cols["speed"]]])

# 🔥 임베딩 인덱스
road_col = cols["road"]
section_col = cols["section"]

road_to_idx = {v: i for i, v in enumerate(full_df[road_col].unique())}
section_to_idx = {v: i for i, v in enumerate(full_df[section_col].unique())}

# 🔥 추가 성능 팁: road+section 결합
full_df["road_section"] = full_df[road_col] + "_" + full_df[section_col]
road_section_to_idx = {v: i for i, v in enumerate(full_df["road_section"].unique())}

print("fit 완료!")

# 메모리 해제
del full_df
del all_data

# =========================
# 🔥 2단계: chunk 단위 처리
# =========================
print("=== 파일별 전처리 시작 ===")

chunk_size = 100000

for file in csv_files:
    print(f"처리중: {file}")

    output_file = os.path.join(output_folder, os.path.basename(file))
    first = True

    for chunk in pd.read_csv(file, chunksize=chunk_size, dtype=str):

        # 컬럼 매핑
        chunk_cols = {k: chunk.columns[v] for k, v in COL_IDX.items()}

        # =========================
        # 🔥 숫자형 변환
        # =========================
        for key in ["traffic", "speed", "rain", "snow", "visibility"]:
            chunk[chunk_cols[key]] = pd.to_numeric(chunk[chunk_cols[key]], errors='coerce').fillna(0)

        # =========================
        # 🔥 log 변환
        # =========================
        chunk[chunk_cols["rain"]] = np.log1p(chunk[chunk_cols["rain"]])
        chunk[chunk_cols["snow"]] = np.log1p(chunk[chunk_cols["snow"]])
        chunk[chunk_cols["traffic"]] = np.log1p(chunk[chunk_cols["traffic"]])

        # =========================
        # 🔥 스케일링
        # =========================
        chunk[[chunk_cols["rain"], chunk_cols["snow"]]] = scaler_minmax_rain.transform(
            chunk[[chunk_cols["rain"], chunk_cols["snow"]]]
        )

        chunk[[chunk_cols["visibility"]]] = scaler_minmax_vis.transform(
            chunk[[chunk_cols["visibility"]]]
        )

        chunk[[chunk_cols["traffic"], chunk_cols["speed"]]] = scaler_std.transform(
            chunk[[chunk_cols["traffic"], chunk_cols["speed"]]]
        )

        # =========================
        # 🔥 임베딩 인덱스
        # =========================
        chunk["road_idx"] = chunk[chunk_cols["road"]].map(road_to_idx).fillna(-1).astype(int)
        chunk["section_idx"] = chunk[chunk_cols["section"]].map(section_to_idx).fillna(-1).astype(int)

        # 🔥 추가 성능 팁
        chunk["road_section"] = chunk[chunk_cols["road"]] + "_" + chunk[chunk_cols["section"]]
        chunk["road_section_idx"] = chunk["road_section"].map(road_section_to_idx).fillna(-1).astype(int)

        # =========================
        # 🔥 저장
        # =========================
        chunk.to_csv(
            output_file,
            mode='w' if first else 'a',
            index=False,
            header=first,
            encoding='utf-8-sig'
        )

        first = False

    print(f"완료: {output_file}")

print("전체 완료!")